In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
torch.manual_seed(math.random.randint(1,10000))
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
droupout = 0.1

In [ ]:
def scaled_dot_attention(q,k,v, mask = none):
  d_k = q.size(-1)
  attention_score = (q@k.transpose(-2,-1))/math.sqrt(d_k)
  if mask is not None:
    mask = attention_score.masked_fill(mask == 0,-1e9)
  attention_score = F.softmax(attention_score, dim = -1)
  output = attention_score@v
  return output, attention_score



In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, num_heads, dropout):
    super().__init__()
    self.d_model = d_model
    self.num_heads = num_heads
    self.d_k = d_model//num_heads
    self.q_linear = nn.Linear(d_model, d_model)
    self.k_linear = nn.Linear(d_model, d_model)
    self.v_linear = nn.Linear(d_model, d_model)
    self.out = nn.Linear(d_model, d_model)


  def forward(self,q,k,v):
    batch_size = q.size(0)
    q = self.q_linear(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)
    k = self.k_linear(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)
    v = self.v_linear(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)

    scores, attention_prob = scaled_dot_attention(q,k,v)
    concat = scores.transpose(1,2).contiguos().view(batch_size, -1, self.d_model)
    output = self.out(concat)

    return output


In [ ]:
class FeedForward(nn.Module):
  def __init__(self, d_model d_ff, dropout):
    super().__init__()
    self.linear1 = nn.Linear(d_model, d_ff)
    self.linear2 = nn.Linear(d_ff, d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self,x):
    x = self.linear1(x)
    x = F.relu(x)
    x = self.dropout(x)
    x = self.linear2(x)
    return x
